<!-- 1. 研究背景与动机
   - 多模态现状: LLM(GPT-3/ChatGPT/Vicuna)能力强，传统VLM(BLIP-2/Kosmos-1)受限于弱LLM
   - GPT-4启发：展现高级多模态能力但技术未公开，推测核心是"强LLM+视觉对齐"
   - 核心问题：极简架构实现视觉-LLM对齐、复现GPT-4能力、解决语言生成质量问题
   - 研究贡献：极简架构、两阶段训练、emergent能力、开源生态
2. 模型架构（极简设计）：
   - 核心组件1: 视觉编码器（复用BLIP-2，含ViT-G/14+Q-Former，冻结）
   - 核心组件2: 线性投影层（唯一可训练，连接视觉特征与LLM嵌入空间）
   - 核心组件3:语言模型（Vicuna，基于LLaMA，达ChatGPT90%性能，冻结）
   - 架构决策：冻结预训练组件、无复杂跨模态模块、开源可复现
3. 两阶段训练流程：
    - 阶段一：预训练对齐
      - 目标：建立视觉-语言基础关联
      - 数据：LAION+Conceptual Captions+SBU（约500万对）
      - 配置：4 x A100, batch=256， 20k步，10小时
      - 问题：语言生成重复、碎片化
    - 阶段二：精细调优
      - 数据构建：5k图生成描述 -> ChatGPT清洗 -> 人工验证 -> 3.5k高质量对
      - 训练模版：###Human: <图像> 指令 ###Assistant: 回答
      - 配置：1 x A100, batch=12，400步，7分钟
      - 效果：解决生成质量问题，提升可用性
4. 核心能力（emergent特性）：
   - 基础视觉理解：详细图像描述、视觉现象解释
   - 创造性生成：图像灵感诗歌/故事/广告、手写草稿转网站
   - 实用功能：食物食谱生成、图像事实检索、植物病害诊断
   - 复杂语义理解：meme幽默解读、深层场景语义分析
5. 实验验证：
   - 定性对比：8任务对比BLIP-2，MiniGPT-4在复杂任务优势显著；两阶段训练前后生成质量提升
   - 定量评估：
     - 高级任务：meme/食谱/广告/诗歌（MiniGPT-4平均65%成功率，BLIP-2仅5%）
     - COCO描述：MiniGPT-4准确率66.2%，BLIP-2 27.5%
     - 调优效果：详细描述/诗歌生成失败率从35%/32％降至2%/1%
   - 消融实验
     - 架构变体：移除Q-Former影响小，多线性层/微调Q-Former性能下降
     - BLIP-2微调：用相同数据微调仍无法完成复杂任务
     - 数据集对比：Localized Narratives数据集导致表达单调、泛化差

6. 局限性与未来方向：
- 局限性：
  - 幻觉问题：生成越长幻觉率越高（CHAIR_i指标：长描述9.6，短描述7.2）
  - 空间理解不足：难以定位物体、判断空间关系
  - 传统基准弱：AOK-VQA/GQA初始分低于BLIP-2 （58.2/32.2 vs 80.2/42.4）
- 未来方向：RLHF+幻觉检测、空间对齐数据集、优化传统基准性能、高效跨模态对齐

7. 核心启示
- 核心逻辑：强LLM能力+简单视觉对齐->高级多模态能力
- 训练启示：少量高质量数据微调可大幅提升生成质量
- 应用价值：开源范式支撑多模态研究与工业应用 -->

### 1. 引言 Introduction

- LLMs呈现非常丰富的涌现(emergent abilities)能力。
- 推测这样的能力也可以扩展到多模态领域，并推测这是GPT-4强的的视觉描述能力的基础。

- 推出了MiniGPT-4。
  - LLM采用了先进的大语言模型Vicuna(基于LLaMA微调)。
  - 视觉感知方面使用了BLIP-2的预训练组件，包括EVA-CLIP中的ViT-G/14和Q-former。
  - 增加了单一投影层，用于对齐编码的视觉特征和Vicuna，并冻结了其他所有的视觉和语言组件。
  - 最初训练步骤为20,000步，使用4张A100(bathsize=256)
  - 图片注解数据集：LAION, Conceptual Captions, SBU, 
  - 只是对齐还不够(理解图像内容)，另外收集了3500对详细的图像描述，进一步微调模型。
  - 采用设计的模版，提升自然语言生成的自然性和可用性。

<div style="background-color:#f9f9f9; padding:10px; border-radius:5px; width:80%; margin:auto">
    <image src="./assets/overview.png" />
    <span style="font-size:12px; color:#555;">图1. MiniGPT-4结构。由一个预训练的ViT和Q-Former视觉编码器、一个线性投影层以及一个先进的Vicuna大型语言模型组成。MiniGPT-4仅需训练线性投影层，即可将视觉特征与Vicuna对齐。</span>
</div>

__主要发现__
- 有力证明了对齐视觉特征和先进的LLM可以实现强大的视觉-语言能力。
- 只有一个线性投影层就可以有效对齐，MiniGPT-4只需要10小时(4 A100)
- 仅用图像-短注解对不足以开发出高性能的模型，通过少量的图像-详细描述对微调可以显著提升。

### 2. 相关工作 Related Work

### 3. 方法论 Method

- 使用Vicuna(基于LLaMA微调)作为语言解码器
- 使用BLIP-2相同的视觉编码器，即ViT与其预训练的Q-Former相结合
- 通过线性投影层弥合视觉编码器和LLM之间的差距

- 两阶段的训练方法：
  1. 在大量的对齐图像-文本对数据集上预训练
  2. 使用少量但高质量的图像-文本和对话模版数据集进行微调

<a id="pretraining_stage"></a>
#### 3.1 第一阶段的预训练阶段 First pretraining stage

预训练阶段，视觉编码器和LLM保持冻结。只有线性投影层被预训练。
- 使用Conceptual Caption，SBU和LAION来训练模型。
- 20,000训练步骤(batchsize 256) - 500万图像-文本对 (10小时，4 A100(80G))


__一阶段预训练问题__

- MiniGPT-4 展现出对人类查询合理的恢复。但是它有时也会产生不连贯的语言输出，如重复的词语或句子、支离破碎的句子或无关内容。
- 这个问题和GPT3遇到的挑战类似。（通过指令微调->GPT-3.5解决）


#### 3.2 为视觉语言领域构建高质量的对齐数据集 Curating a high-quality alignment dataset for vision-language domain.

- LLMs有很多
  - 指令微调数据集(Instructed Finetuning Datasets):
  - 会话数据集(Conversation)

__初始对齐图像-文本生成__

- 用第一阶段预训练得出的模型，生成输入图像的全面描述。遵循如下的Vicuna语言模型会话格式的提示

```
###Human: <Img><ImageFeature></Img>Describe this image in detail. Give as many details as possible. Say everything you see. ###Assistant:
```

- `<ImageFeature>`代表线性投影层产生的视觉特征。

- 为了识别不完整的句子，检查句子是否超过80个词。如果没有，则加入额外提示`###Human: Continue ###Assistant:`，提示模型延长输出。合并输出，创造更全面的图像描述。从COCO数据集中选择了5000张图片，利用 [预训练](#pretraining_stage) 的模型为每张图像生成对应的语言描述。


__数据后处理 Data post-processing__

上述自动生成的描述包含噪声或不连贯。（如句子重复、碎片化或无关内容），我们利用ChatGPT通过以下提示来修正描述：

```
Fix the error in the given paragraph. Remove any repeating sentences, meaningless characters, not English sentences, and so on. Remove unnecessary repetition. Rewrite any incomplete sentences. Return directly the results without explanation. Return directly the input paragraph if it is already correct without explanation.
```

- 然后手动核实描述内容的正确性。
  - 具体来说，首先识别了几个经常的错误："I'm sorry I made a mistake ..."或"I apologize for that ..."
  - 然后通过硬编码规则自动过滤这些错误。
  - 手动优化生成的描述，剔除ChatGPT未能识别的重复词语或句子。
  - 5000对中大约有3500对满足要求。用于二阶段的微调。

#### 3.3 第二阶段微调

- 在第二阶段，使用高质量图像-文本对对模型进行微调。使用以下模版中的预定义提示：

```
###Human: <Img><ImageFeature></Img><Instruction>###Assistant:
```

- `<Instruction>`是从预定义指令集中随机抽样的指令。比如：
  - "Describe this image in detail" 
  - "Counld you describe the contents of this image for me"
  - 我们不计算该特定的文本-图像提示词的损失

MiniGPT-4能够产生更自然、更可靠的语言输出。仅400步，batchsize为12，单A100 GPU约7分钟。

### 4. 实验 Experiments

- 实验中MiniGPT-4展示多样的emergent能力，例如：
    1. 生产详细的图像描述
    2. 识别表情包中的有趣元素
    3. 根据照片提供食物食谱
    4. 为图片写诗歌
    5. 图像注解的定量结果


#### 4.1 通过定性实例揭示MiniGPT-4的涌现能力 Uncovering emergent abilities with MiniGPT-4 through qualitative examples

<div style="
    background-color:#f9f9f9; 
    padding:10px; 
    border-radius:5px; 
    width:90%; 
    margin: auto;
    display:flex;
    justify-content:center;
    gap: 5px;
    align-items:center;
">
    <div width="50%" style="text-align:center;">
        <image src="./assets/fig5.png" />
        <span style="font-size:12px; color:#555;">图2. 详细描述。</span>
    </div>
    <div width="50%" style="text-align:center;">
        <image src="./assets/fig3.png" />
        <span style="font-size:12px; color:#555;">图3. 广告生成。</span>
    </div>
</div>

#### 4.2 定量分析 Quantitative analysis

__高级能力 Advanced Abilities__

- 构建4个任务的数据集(100张，每个任务25张)：
    1. 解释表情包笑点: "Explain why this meme is funny."
    2. 食谱生成: "How should I make something like this?"
    3. 广告创作: "Help me draft a professional advertisement for this."
    4. 诗歌创作: "Can you craft a beautiful poem about this image?"
- 使用人工评估员(human evaluators)评估，并与BLIP-2进行对比：

| | Meme | Recipes | Ads | Poem | Avg. |
|---|---|---|---|---|---|
| BLIP-2 | 0/25 | 4/25 | 1/25| 0/25 | 5/100 |
| MiniGPT-4 | 8/25 | 18/25 | 19/25 | 20/25 | 65/100 |

__图像注解 Image Captioning__

- 评估了MiniGPT-4在COCO注解基准测试中的表现，和BLIP-2进行了对比。（我们的模型生成了更丰富的视觉细节，因此传统的基于相似度的图像-注解评估标准不行）使用ChatGPT检查生成内容是否涵盖所有真实信息。结果如下：

- Table2: COCO数据集上的评估，使用ChatGPT来判断生成的注解是否包含所有的可视物体并和实际的注解一致。

| | BLIP-2 | MiniGPT-4 |
|---|---|---|
|Correctness | 1376/5000 | 3310/5000 |
| Percentage | 27.5% | 66.2% |

#### 4.3 第二阶段的微调分析 Analysis on the second-stage finetuning

__第二阶段微调的有效性 Effectiveness of the second-stage finetuning__
- 从图5可以看出，仅通过第一阶段的预训练，模型生成内容会有错误，如：
  - 重复内容
  - 碎片化的句子
  - 无关内容
- 通过第二阶段的微调，问题得到缓解。

<div style="
    background-color:#f9f9f9; 
    padding:10px; 
    border-radius:5px; 
    width:90%; 
    margin: auto;
    display:flex;
    justify-content:center;
    gap: 5px;
    align-items:flex-end;
">
    <div width="50%" style="text-align:center;">
        <image src="./assets/ablation.png" />
        <span style="font-size:12px; color:#555;">图5: 在二阶段微调前MiniGPT-4无法生成完整的文本。微调之后内容有所提升。</span>
    </div>
    <div width="50%" style="text-align:center;">
        <image src="./assets/failure.png" />
        <span style="font-size:12px; color:#555;">图6: 一个例子展示MiniGPT-4的限制。MiniGPT-4会幻觉出不存在的桌布，并且无法准确定位窗户的位置。</span>
    </div>
</div>



- 为了量化二阶段微调的影响，从COCO测试集中随机抽样了100张图像，并研究模型在俩个任务上的表现：
  - 详细描述生成，"Describe the image in detail."
  - 诗歌创作，"Can you write a beautiful poem about this image?"
- 手工统计了模型的失败次数，结果如下：

- Table3: 细节注解和诗歌生成任务在阶段2微调之前和之后的失败率

| Failure rate | Detailed caption | Poem |
|---|---|---|
| Before stage-2 | 35% | 32% |
| After stage-2 | 2% | 1% |

__原始的BLIP-2能否从第二阶段的数据中收益 Can the original BLIP-2 benefit from the second-stage data?__

- 使用第二阶段的数据对BLIP-2进行微调。
  - 注意两个模型使用相同的视觉模块
  - 但BLIP-2的语言模型是FlanT5XXL（相比于Vicuna弱一点）
- 结果：BLIP-2 FT仍生成简短的响应，且无法推广到诸如表情包解释和网站编码等高级任务。==> 更高级 LLM 在 VLM 系统中的有效性。

__使用LN数据集的第二阶段 Second stage with Localized Narratives__

- 数据集Localized Narratives是一个详细的图像描述数据集，注释者在描述图像的同时对响应区域畸形了定位。
- MiniGPT-4 LocNa则是将第二阶段替换成LN数据集。
- 然而输出质量低，详见[下图](#fig_2)

<a id="fig_2"></a>
<div style="
    background-color:#f9f9f9; 
    padding:10px; 
    border-radius:5px; 
    width:90%; 
    margin: auto;
    display:flex;
    justify-content:center;
    gap: 5px;
    align-items:flex-end;
">
    <div width="50%" style="text-align:center;">
        <image src="./assets/rebuttal_meme.png" />
        <span style="font-size:12px; color:#555;">(a) 表情包解释</span>
    </div>
    <div width="50%" style="text-align:center;">
        <image src="./assets/rebuttal_web.png" />
        <span style="font-size:12px; color:#555;">(b) 网络创造</span>
    </div>
</div>

#### 4.4 模型结构上的消融实验 Ablation on the architecture designs

- 使用了下面几个不同架构设计进行了实验：
  1) 移除Q-Former并直接将VIP输出映射到Vicuna的嵌入空间
  2) 使用三层线性层而非一层
  3) 额外微调Q-Former

- 结论：
  1) MiniGPT-4 w/o Q-Former性能相似（见上图），说明了BLIP-2中的Q-Former并没有作为关键的角色
  2) MiniGPT-4 + 3 Layers 和
  3) MiniGPT-4 + finetuning Q-Former性能稍微下降，说明了单个线性投影层似乎足够对齐视觉编码器和大语言模型的（在我们构建的有限的数据集上）

- 表4: 不同架构设计的消融实验结果

| Model | AOK_VQA | GQA |
|---|---|---|
| MiniGPT-4 | 58.2 | 32.2 |
| MiniGPT-4 w/o Q-Former | 56.9 | 33.4 |
| MiniGPT-4 + 3 Layers | 49.7 | 31.0 |
| MiniGPT-4 + Finetune Q-Former | 52.1 | 28.0 |

#### 4.5 局限性分析 Limitation analysis

- 由于MiniGPT-4在LLMs上构建，所以继承了它们的一些局限性：
  - 幻觉(Hallucination): 如在图6中，错误识别了不存在的白色桌布。

- 使用CHAIR_i指标来衡量生成的幻觉率。
- 通过两个提示控制生成长度：
  - MiniGPT-4 (long): Please describe this image as detailed as possible. 
  - MiniGPT-4 (short): Please describe the image shortly and precisely, in less than 20 words.
- 结果：更长的文本更容易产生幻觉。

- 潜在的解决方案：使用AI幻觉检测模块的强化学习

- 表5: 幻觉率评估

| | CHAIR_i | Avg.Length |
|---|---|---|
| BLIP-2 | 1.3 | 6.5 |
| MiniGPT-4 (short) | 7.2 | 28.8 |
| MiniGPT-4 (long) | 9.6 | 175 |

__空间信息理解 Spatial Information Understanding__

- MiniGPT-4的视觉感知有限，难以区分空间定位。(如在图6中，无法准确定位窗户的位置)
- 潜在的解决方案：在RefCOCO或Visual Genome等数据集上进行训练

### 5. 讨论 Discussion